# Diffusion-LM on E2E NLG — Colab Pipeline
**CENG 467 Term Project — Zübeyr Almaho (300201023)**

Bu notebook full pipeline'ı çalıştırır:
1. Repo clone + dependencies
2. E2E NLG dataset cache
3. GPT-2 baseline eğitim
4. T5 baseline eğitim
5. Diffusion-LM eğitim
6. Generation + metrikler
7. Ablation çalıştırmaları
8. Sonuç paketleme (`results.zip` indir)

**Runtime → Change runtime type → A100 (varsa) yoksa T4.**

**Süre (A100):** ~3 saat full pipeline. **(T4):** ~7 saat.

Zamanın kısıtlıysa Bölüm 5-7'yi atla; 2-4 progress report için yeter.

## 0. GPU kontrolü

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 1. Repo + dependencies

In [ ]:
import os
if not os.path.isdir('diffusion-lm-ctg'):
    !git clone https://github.com/zubeyralmaho/diffusion-lm-ctg.git
%cd diffusion-lm-ctg
!git pull --rebase   # repo varsa son commit'leri çek
!pip install -q -r requirements.txt

## 1b. Smoke test (10 sn)

3 saatlik eğitime başlamadan önce pipeline'ın çalıştığını doğrula. CPU'da çalışır, model indirmez. 4 testin de PASS olması lazım.

In [ ]:
!python -m tests.smoke_test

## 2.5 Sanity baseline: untrained GPT-2 (3-5 dk)

Progress report için gerçek bir "lower bound" rakamı üretir. Hiç fine-tune etmeden, pretrained `gpt2`'yi `--pretrained` flag'i ile direkt generate.py'ye veriyoruz, test setinden 200 örnek üzerinde metrikleri hesaplıyoruz. BLEU çok düşük çıkacak (~1-5) ama bu \emph{gerçek}, \emph{meşru} bir sanity baseline.

Bu hücreden gelen JSON çıktısının içeriğini Zübeyr'e at — rapora satır olarak ekleyecek.

In [ ]:
!python -m src.generate --config configs/gpt2_baseline.yaml \
    --split test --limit 200 --pretrained \
    --out results/generations/gpt2_pretrained_sanity.jsonl
!python -m src.evaluate \
    --predictions results/generations/gpt2_pretrained_sanity.jsonl \
    --out results/metrics/gpt2_pretrained_sanity.json
!cat results/metrics/gpt2_pretrained_sanity.json

## 2. Dataset cache

In [ ]:
!bash scripts/prepare_data.sh

## 3. GPT-2 baseline — eğitim + generation + eval

In [ ]:
!python -m src.train --config configs/gpt2_baseline.yaml
!python -m src.generate --config configs/gpt2_baseline.yaml --split test \
    --out results/generations/gpt2_baseline.jsonl
!python -m src.evaluate \
    --predictions results/generations/gpt2_baseline.jsonl \
    --out results/metrics/gpt2_baseline.json
!cat results/metrics/gpt2_baseline.json

## 4. T5 baseline — eğitim + generation + eval

In [ ]:
!python -m src.train --config configs/t5_baseline.yaml
!python -m src.generate --config configs/t5_baseline.yaml --split test \
    --out results/generations/t5_baseline.jsonl
!python -m src.evaluate \
    --predictions results/generations/t5_baseline.jsonl \
    --out results/metrics/t5_baseline.json
!cat results/metrics/t5_baseline.json

## 5. Diffusion-LM (proposed method) — eğitim + generation + eval

En uzun adım. A100'de ~45 dk, T4'te 2-3 saat. Her epoch sonunda checkpoint kaydediliyor; runtime düşse bile devam edebilirsin.

In [ ]:
!python -m src.train --config configs/diffusion_lm.yaml
!python -m src.generate --config configs/diffusion_lm.yaml --split test \
    --out results/generations/diffusion_lm.jsonl
!python -m src.evaluate \
    --predictions results/generations/diffusion_lm.jsonl \
    --out results/metrics/diffusion_lm.json
!cat results/metrics/diffusion_lm.json

## 6. Karşılaştırma tablosu

Üç modelin metriklerini tek tabloda birleştir.

In [ ]:
!python -m src.compare_results --metrics_dir results/metrics --out results/summary.md
from IPython.display import Markdown; Markdown(open('results/summary.md').read())

## 7. Ablation studies (opsiyonel — uzun)

5 ablation: cosine schedule, DDIM 50/100 steps, prefix length 16/48. Sampling-only ablation'lar base checkpoint'i yeniden kullanır; retraining ablation'ları yeni model eğitir.

Zamanın yoksa bu hücreyi atla — Zübeyr ikinci oturumda yapar.

In [ ]:
!bash scripts/run_ablations.sh

## 8. Sonuçları paketle ve indir

Bu hücre `results.zip` üretir ve Colab'dan tarayıcına indirir. Bu zip'i Zübeyr'e gönder.

In [ ]:
import shutil
shutil.make_archive('results', 'zip', 'results')
from google.colab import files
files.download('results.zip')
print('results.zip hazır — indir ve gönder.')

## 9. Checkpoint'leri Drive'a kopyala (opsiyonel)

Diffusion-LM checkpoint'i büyük (~200MB). Eğer Zübeyr daha sonra ablation/inference yapmak isterse Drive'da saklamak iyi. Berkay'ın Drive'ına gidecek, sonra paylaşılır.

In [ ]:
# Sadece bu hücreyi çalıştırırsan Drive'a kopyalar. Drive'ı bağlamak için izin isteyecek.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/diffusion-lm-ctg-ckpts
!cp -r checkpoints/* /content/drive/MyDrive/diffusion-lm-ctg-ckpts/
print('Checkpoint\'ler Drive\'a kopyalandı.')